<a href="https://colab.research.google.com/github/gilbertoag2007/fiap-tech-challenge-fase3/blob/main/tech_challenge_fase_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#AGRUPANDO AS 5 PARTES DO DATASET

In [1]:
# Clone do repositório no GITHUB
!git clone https://github.com/gilbertoag2007/fiap-tech-challenge-fase3.git

Cloning into 'fiap-tech-challenge-fase3'...
remote: Enumerating objects: 89, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 89 (delta 4), reused 0 (delta 0), pack-reused 80 (from 1)
Receiving objects: 100% (89/89), 261.06 MiB | 22.55 MiB/s, done.
Resolving deltas: 100% (41/41), done.


In [2]:
# ============================================================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

import pandas as pd
import re
import spacy
from pathlib import Path

!pip install -q pandas pyarrow openpyxl
!pip install pandas openpyxl spacy
!python -m spacy download pt_core_news_lg



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 1.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# CARREGAMENTO DOS DADOS

In [3]:
"""
===========================================================================
OBJETIVO
===========================================================================

Este script reúne vários arquivos XLSX que foram divididos devido ao
limite de tamanho do GitHub (ex.: arquivos maiores que 80 MB).

Ao final será gerado um único dataset que servirá de entrada para as
etapas de:

1. Identificação de dados pessoais (PHI)
2. Anonimização
3. Limpeza dos dados
4. Fine-Tuning de uma LLM

Bibliotecas necessárias:

pip install pandas openpyxl

===========================================================================
"""

# ==========================================================================
# CONFIGURAÇÕES
# ==========================================================================

# Pasta onde estão os arquivos divididos
PASTA_DADOS = "/content/fiap-tech-challenge-fase3/data"

# Nome do arquivo de saída
ARQUIVO_FINAL = "dataset_medico_completo.xlsx"

# ==========================================================================
# Localiza todos os arquivos XLSX da pasta
# ==========================================================================

print("=" * 70)
print("LOCALIZANDO ARQUIVOS...")
print("=" * 70)

# Procura todos os arquivos .xlsx
arquivos = sorted(Path(PASTA_DADOS).glob("*.xlsx"))

# Verifica se encontrou arquivos
if len(arquivos) == 0:
    raise Exception("Nenhum arquivo XLSX encontrado.")

print(f"Foram encontrados {len(arquivos)} arquivos.\n")

# ==========================================================================
# Leitura dos arquivos
# ==========================================================================

print("=" * 70)
print("LENDO ARQUIVOS...")
print("=" * 70)

lista_dataframes = []

for arquivo in arquivos:

    print(f"Lendo {arquivo.name}")

    # Carrega o arquivo para um DataFrame
    df = pd.read_excel(
        arquivo,
        engine="openpyxl"
    )

    # Guarda o DataFrame em memória
    lista_dataframes.append(df)

# ==========================================================================
# Junta todos os DataFrames
# ==========================================================================

print("\nUnindo arquivos...")

dataframe_original = pd.concat(
    lista_dataframes,
    ignore_index=True
)

print("Arquivos unidos com sucesso!")



LOCALIZANDO ARQUIVOS...
Foram encontrados 4 arquivos.

LENDO ARQUIVOS...
Lendo dataset_medico_part001.xlsx
Lendo dataset_medico_part002.xlsx
Lendo dataset_medico_part003.xlsx
Lendo dataset_medico_part004.xlsx

Unindo arquivos...
Arquivos unidos com sucesso!


In [4]:
# ==========================================================================
# Limpeza básica
# ==========================================================================

print("\nRealizando limpeza inicial...")

# Remove linhas totalmente vazias
dataframe_original.dropna(
    how="all",
    inplace=True
)

# Remove registros duplicados
dataframe_original.drop_duplicates(
    inplace=True
)

# Reinicia a numeração do índice
dataframe_original.reset_index(
    drop=True,
    inplace=True
)



Realizando limpeza inicial...


In [5]:
# ==========================================================================
# REDUZ O TAMANHO DO DATASET PARA AGILIZAR OS TESTES EM DESENVOLVIMENTO
# ==========================================================================

dataframe_reduzido = dataframe_original.sample(frac=0.01, random_state=42)


In [6]:
# ==========================================================================
# Informações do dataset
# ==========================================================================

print("\nResumo do dataset")

print("-" * 60)

print(f"Quantidade de registros : {len(dataframe_reduzido):,}")

print(f"Quantidade de colunas   : {len(dataframe_reduzido.columns)}")

print("\nColunas encontradas:\n")

for coluna in dataframe_reduzido.columns:
    print(f" - {coluna}")

# ==========================================================================
# Verificação de valores nulos
# ==========================================================================

print("\nValores ausentes por coluna\n")

print(dataframe_reduzido.isnull().sum())

# ==========================================================================
# Estatísticas básicas
# ==========================================================================

print("\nPrimeiros registros:\n")

print(dataframe_reduzido.head())


Resumo do dataset
------------------------------------------------------------
Quantidade de registros : 3,841
Quantidade de colunas   : 6

Colunas encontradas:

 - id
 - pergunta
 - resposta
 - condicao
 - especialidade_medica
 - tipo_pergunta

Valores ausentes por coluna

id                      0
pergunta                0
resposta                0
condicao                0
especialidade_medica    0
tipo_pergunta           0
dtype: int64

Primeiros registros:

            id                                           pergunta  \
56799   331651             Demência. O que pode levar à demência?   
299108  429542  Meu namorado disse que há um tempo atrás foi d...   
172048  338711  Estou em um relacionamento relativamente recen...   
21162    27410  O que pode causar zumbido pulsátil , sendo que...   
82076   117978  exercício físico diminui a glicemia e a glicos...   

                                                 resposta  \
56799   O que pode levar à demência:Resposta: fator id..

In [7]:

# ==========================================================================
# Salva o dataset consolidado
# ==========================================================================

print("\nSalvando arquivo consolidado...")

dataframe_reduzido.to_excel(
    ARQUIVO_FINAL,
    index=False,
    engine="openpyxl"
)

print("\nArquivo salvo com sucesso!")

print(f"\nArquivo gerado: {ARQUIVO_FINAL}")



Salvando arquivo consolidado...

Arquivo salvo com sucesso!

Arquivo gerado: dataset_medico_completo.xlsx


# DECTECTAR PHI - Protected Health Information

In [13]:
# VERSÃO ATUAL

import re
import pandas as pd
import spacy


# ============================================================
# 1. CARREGAR MODELO NLP
# ============================================================

nlp = spacy.load("pt_core_news_lg")


# ============================================================
# 2. TERMOS MÉDICOS
# ============================================================

TERMOS_MEDICOS = {
    "esclerose",
    "esquizofrenia",
    "bipolar",
    "bipolaridade",
    "artrose",
    "discopatias",
    "discopatia",
    "diagnóstico",
    "diagnostico",
    "paciente",
    "pacientes",
    "médico",
    "medico",
    "médica",
    "medica",
    "doença",
    "doenca",
    "doenças",
    "doencas",
    "neurônio",
    "neuronio",
    "cirurgia",
    "cirúrgico",
    "cirurgico",
    "consulta",
    "teleconsulta",
    "tratamento",
    "procedimento",
    "procedimentos",
    "hospital",
    "clínica",
    "clinica",
    "câncer",
    "cancer",
    "diabetes",
    "hipertensão",
    "hipertensao",
    "depressão",
    "depressao",
    "ansiedade",
    "herpes",
    "zoster",
    "herpes zoster",
}


# ============================================================
# 3. PADRÕES REGEX
# ============================================================

PADROES = {

    # CPF
    "CPF": re.compile(
        r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b"
    ),

    # E-mail
    "EMAIL": re.compile(
        r"\b[A-Za-z0-9._%+-]+@"
        r"[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
    ),

    # Telefone
    "TELEFONE": re.compile(
        r"(?<!\d)"
        r"(?:\+55\s?)?"
        r"(?:\(?\d{2}\)?\s?)?"
        r"(?:9\d{4}|\d{4})[-\s]?\d{4}"
        r"(?!\d)"
    ),

    # CEP
    "CEP": re.compile(
        r"\b\d{5}-?\d{3}\b"
    ),

    # Data
    "DATA": re.compile(
        r"\b(?:0?[1-9]|[12]\d|3[01])"
        r"[/\-]"
        r"(?:0?[1-9]|1[0-2])"
        r"[/\-]"
        r"(?:19|20)\d{2}\b"
    ),

    # RG
    "RG": re.compile(
        r"\b\d{1,2}\.?\d{3}\.?\d{3}-?[0-9Xx]\b"
    ),

    # Estado civil
    "ESTADO_CIVIL": re.compile(
        r"\b(?:"
        r"solteiro|solteira|"
        r"casado|casada|"
        r"vi[uú]vo|vi[uú]va|"
        r"divorciado|divorciada|"
        r"separado|separada|"
        r"uni[aã]o\s+est[aá]vel"
        r")\b",
        re.IGNORECASE
    )
}


# ============================================================
# 4. VALIDAÇÃO DE CPF
# ============================================================

def cpf_valido(cpf):

    numeros = re.sub(r"\D", "", cpf)

    # CPF deve possuir 11 dígitos
    if len(numeros) != 11:
        return False

    # Rejeita sequências como 11111111111
    if numeros == numeros[0] * 11:
        return False

    # Primeiro dígito verificador
    soma = sum(
        int(numeros[i]) * (10 - i)
        for i in range(9)
    )

    resto = soma % 11
    digito1 = 0 if resto < 2 else 11 - resto

    if int(numeros[9]) != digito1:
        return False

    # Segundo dígito verificador
    soma = sum(
        int(numeros[i]) * (11 - i)
        for i in range(10)
    )

    resto = soma % 11
    digito2 = 0 if resto < 2 else 11 - resto

    return int(numeros[10]) == digito2


# ============================================================
# 5. DETECTAR PADRÕES REGEX
# ============================================================

def detectar_regex(texto):

    encontrados = set()

    for tipo, padrao in PADROES.items():

        ocorrencias = padrao.findall(texto)

        # CPF exige validação dos dígitos verificadores
        if tipo == "CPF":

            if any(cpf_valido(ocorrencia) for ocorrencia in ocorrencias):
                encontrados.add(tipo)

        # Demais padrões são considerados encontrados
        elif ocorrencias:

            encontrados.add(tipo)

    return encontrados


# ============================================================
# 6. DETECTAR NOME DE PESSOA
# ============================================================

def detectar_pessoa(entidade):

    valor = entidade.text.strip()
    valor_lower = valor.lower()

    # Evita que termos médicos sejam classificados como nomes
    if valor_lower in TERMOS_MEDICOS:
        return False

    palavras = valor.split()

    # Nome deve possuir pelo menos duas palavras
    if len(palavras) < 2:
        return False

    # Pelo menos duas palavras devem começar com letra maiúscula
    palavras_maiusculas = sum(
        palavra[:1].isupper()
        for palavra in palavras
    )

    return palavras_maiusculas >= 2


# ============================================================
# 7. DETECTAR LOCALIZAÇÃO
# ============================================================

def detectar_localizacao(entidade, texto):

    valor = entidade.text.strip()
    valor_lower = valor.lower()

    # Evita termos médicos
    if valor_lower in TERMOS_MEDICOS:
        return False

    padroes_contexto = [
        r"\bresidente em\b",
        r"\bmora em\b",
        r"\bmorador de\b",
        r"\bnatural de\b",
        r"\bnascido em\b",
        r"\bnascida em\b",
        r"\bproveniente de\b",
        r"\bprocedente de\b",
        r"\bendereço\b",
        r"\bdomicílio\b",
        r"\bdomicilio\b",
        r"\bcidade de\b",
        r"\bestado de\b",
        r"\bmunicípio de\b",
        r"\bmunicipio de\b",
        r"\bresidência em\b",
        r"\bresidencia em\b"
    ]

    return any(
        re.search(padrao, texto, re.IGNORECASE)
        for padrao in padroes_contexto
    )


# ============================================================
# 8. DETECTAR ORGANIZAÇÃO
# ============================================================

def detectar_organizacao(entidade, texto):

    valor = entidade.text.strip()
    valor_lower = valor.lower()

    # Evita termos médicos
    if valor_lower in TERMOS_MEDICOS:
        return False

    indicadores = [
        "hospital",
        "clínica",
        "clinica",
        "universidade",
        "instituto",
        "laboratório",
        "laboratorio",
        "empresa",
        "fundação",
        "fundacao",
        "associação",
        "associacao",
        "faculdade",
        "escola",
        "prefeitura",
        "secretaria",
        "ministério",
        "ministerio"
    ]

    # Verifica o próprio nome da organização
    if any(
        indicador in valor_lower
        for indicador in indicadores
    ):
        return True

    # Verifica contexto
    padroes_contexto = [
        r"\bno hospital\b",
        r"\bna clínica\b",
        r"\bna clinica\b",
        r"\bno laboratório\b",
        r"\bno laboratorio\b",
        r"\bda universidade\b",
        r"\bdo instituto\b",
        r"\bna empresa\b",
        r"\bno instituto\b"
    ]

    return any(
        re.search(padrao, texto, re.IGNORECASE)
        for padrao in padroes_contexto
    )


# ============================================================
# 9. DETECTAR ENTIDADES COM CONTEXTO
# ============================================================

def detectar_entidades(texto):

    entidades_detectadas = set()

    doc = nlp(texto)

    for entidade in doc.ents:

        if entidade.label_ == "PER":

            if detectar_pessoa(entidade):
                entidades_detectadas.add("NOME_PESSOA")

        elif entidade.label_ == "ORG":

            if detectar_organizacao(entidade, texto):
                entidades_detectadas.add("ORGANIZACAO")

        elif entidade.label_ in {"LOC", "GPE"}:

            if detectar_localizacao(entidade, texto):
                entidades_detectadas.add("LOCALIZACAO")

    return entidades_detectadas


# ============================================================
# 10. ANALISAR TEXTO
# ============================================================

def analisar_texto(valor):

    # Ignora valores nulos
    if pd.isna(valor):
        return []

    texto = str(valor).strip()

    # Ignora textos vazios
    if not texto:
        return []

    tipos = set()

    # Detecta CPF, e-mail, telefone, CEP, data, RG etc.
    tipos.update(
        detectar_regex(texto)
    )

    # Detecta entidades usando SpaCy + regras contextuais
    tipos.update(
        detectar_entidades(texto)
    )

    return sorted(tipos)

In [15]:
import re
import pandas as pd


# ============================================================
# 1. LOCALIZAR A FRASE QUE CONTÉM O VALOR ENCONTRADO
# ============================================================

def localizar_frase(texto, inicio, fim):

    # Divide o texto em frases considerando ., ! e ?
    frases = re.split(r'(?<=[.!?])\s+', texto)

    posicao_atual = 0

    for frase in frases:

        inicio_frase = posicao_atual
        fim_frase = posicao_atual + len(frase)

        # Verifica se o intervalo da entidade
        # está dentro desta frase
        if inicio >= inicio_frase and fim <= fim_frase:
            return frase.strip()

        posicao_atual = fim_frase + 1

    # Caso não consiga determinar a frase,
    # retorna o texto completo
    return texto.strip()


# ============================================================
# 2. LOCALIZAR REGEX COM O VALOR ENCONTRADO
# ============================================================

def localizar_regex(texto):

    resultados = []

    for tipo, padrao in PADROES.items():

        for ocorrencia in padrao.finditer(texto):

            valor = ocorrencia.group()

            # CPF precisa ser validado
            if tipo == "CPF":

                if not cpf_valido(valor):
                    continue

            resultados.append({
                "entidade": tipo,
                "valor": valor,
                "inicio": ocorrencia.start(),
                "fim": ocorrencia.end()
            })

    return resultados


# ============================================================
# 3. LOCALIZAR ENTIDADES DO SPACY
# ============================================================

def localizar_entidades_spacy(texto):

    resultados = []

    doc = nlp(texto)

    for entidade in doc.ents:

        inicio = entidade.start_char
        fim = entidade.end_char

        valor = entidade.text.strip()

        # --------------------------------------------
        # PESSOA
        # --------------------------------------------

        if entidade.label_ == "PER":

            if detectar_pessoa(entidade):

                resultados.append({
                    "entidade": "NOME_PESSOA",
                    "valor": valor,
                    "inicio": inicio,
                    "fim": fim
                })

        # --------------------------------------------
        # ORGANIZAÇÃO
        # --------------------------------------------

        elif entidade.label_ == "ORG":

            if detectar_organizacao(entidade, texto):

                resultados.append({
                    "entidade": "ORGANIZACAO",
                    "valor": valor,
                    "inicio": inicio,
                    "fim": fim
                })

        # --------------------------------------------
        # LOCALIZAÇÃO
        # --------------------------------------------

        elif entidade.label_ in {"LOC", "GPE"}:

            if detectar_localizacao(entidade, texto):

                resultados.append({
                    "entidade": "LOCALIZACAO",
                    "valor": valor,
                    "inicio": inicio,
                    "fim": fim
                })

    return resultados


# ============================================================
# 4. LOCALIZAR TODAS AS ENTIDADES DO TEXTO
# ============================================================

def localizar_entidades(texto):

    resultados = []

    # --------------------------------------------
    # Regex
    # --------------------------------------------

    resultados.extend(
        localizar_regex(texto)
    )

    # --------------------------------------------
    # SpaCy
    # --------------------------------------------

    resultados.extend(
        localizar_entidades_spacy(texto)
    )

    return resultados


# ============================================================
# 5. GERAR ARQUIVO TXT COM OS RESULTADOS
# ============================================================

def gerar_relatorio_txt(
    dataframe,
    coluna,
    arquivo_saida="registros_localizados.txt"
):

    total_registros = 0

    with open(
        arquivo_saida,
        "w",
        encoding="utf-8"
    ) as arquivo:

        # Cabeçalho
        arquivo.write(
            "RELATÓRIO DE ENTIDADES LOCALIZADAS\n"
        )

        arquivo.write(
            "=" * 80 + "\n\n"
        )

        # --------------------------------------------
        # Percorrer DataFrame
        # --------------------------------------------

        for indice, valor in dataframe[coluna].items():

            # Ignorar valores nulos
            if pd.isna(valor):
                continue

            texto = str(valor).strip()

            if not texto:
                continue

            # ----------------------------------------
            # Localizar entidades
            # ----------------------------------------

            entidades = localizar_entidades(texto)

            if not entidades:
                continue

            # ----------------------------------------
            # Gerar registros
            # ----------------------------------------

            for item in entidades:

                frase = localizar_frase(
                    texto,
                    item["inicio"],
                    item["fim"]
                )

                arquivo.write(
                    f"LINHA: {indice + 1}\n"
                )

                arquivo.write(
                    f"ENTIDADE: {item['entidade']}\n"
                )

                arquivo.write(
                    f"VALOR: {item['valor']}\n"
                )

                arquivo.write(
                    f"FRASE: {frase}\n"
                )

                arquivo.write(
                    "-" * 80 + "\n"
                )

                total_registros += 1

        # --------------------------------------------
        # Rodapé
        # --------------------------------------------

        arquivo.write("\n")
        arquivo.write("=" * 80 + "\n")
        arquivo.write(
            f"TOTAL DE OCORRÊNCIAS: {total_registros}\n"
        )

    print(
        f"Relatório gerado com sucesso: {arquivo_saida}"
    )

    print(
        f"Total de ocorrências localizadas: {total_registros}"
    )

In [17]:
colunas_analisar = [
    "pergunta",
    "resposta"
]

for coluna in colunas_analisar:

    gerar_relatorio_txt(
        dataframe=dataframe_reduzido,
        coluna=coluna,
        arquivo_saida=f"registros_localizados_{coluna}.txt"
    )

Relatório gerado com sucesso: registros_localizados_pergunta.txt
Total de ocorrências localizadas: 823
Relatório gerado com sucesso: registros_localizados_resposta.txt
Total de ocorrências localizadas: 212
